# RAGTruth Baseline Reproduction (Colab)

This notebook prepares RAGTruth baseline data, runs local Hugging Face generation (no Docker TGI), and computes baseline case-level Precision/Recall/F1.

## LongT5 Note

> This notebook reproduces the official RAGTruth baseline detector workflow (prompted hallucination span labeling) using causal LMs.

> For project evaluation with `google/long-t5-tglobal-base` in `gold_context_generation`, use `colab_verifier_eval_ragtruth.ipynb` or `colab_mitigation_eval_ragtruth.ipynb`, which auto-generate `config.colab.yaml` from project defaults.

In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

def run(cmd, cwd=None, check=True, stream=True):
    print(f"\n$ {cmd}")
    if stream:
        process = subprocess.Popen(
            cmd,
            shell=True,
            cwd=str(cwd) if cwd else None,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        out_lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            out_lines.append(line)
        process.wait()
        completed = subprocess.CompletedProcess(
            args=cmd,
            returncode=process.returncode,
            stdout=''.join(out_lines),
            stderr=None,
        )
    else:
        completed = subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
        if completed.stdout:
            print(completed.stdout)

    if completed.returncode != 0:
        if not stream and completed.stderr:
            print(completed.stderr)
        if check:
            raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")
    return completed

REPO_URL = 'https://github.com/xiashuidaolaoshuren/AIST-FYP.git'
REPO_CANDIDATES = [
    Path('/content/AIST-FYP'),
    Path('/content/drive/MyDrive/AIST-FYP'),
    Path('/content/drive/MyDrive/AIST-FYP-main'),
]
BASELINE_REL_PATH = Path('benchmark/RAGTruth/baseline')

REPO_DIR = next((
    p for p in REPO_CANDIDATES if (p / BASELINE_REL_PATH).exists()
), None)
if REPO_DIR is None:
    REPO_DIR = next((p for p in REPO_CANDIDATES if p.exists()), None)

if REPO_DIR is None:
    REPO_DIR = Path('/content/AIST-FYP')
    print('Repository not found in common locations. Cloning into /content/AIST-FYP ...')
    run(f'git clone {REPO_URL} {REPO_DIR}', stream=True)

if not (REPO_DIR / BASELINE_REL_PATH).exists():
    raise FileNotFoundError(
        f'Found repo at {REPO_DIR}, but missing {BASELINE_REL_PATH}. '
        'Please ensure this is the correct AIST-FYP repository.'
    )

DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/AIST-FYP-colab-outputs')
ARTIFACTS_DIR = DRIVE_OUTPUT_DIR / 'ragtruth_baseline'
TRAIN_OUTPUT_DIR = ARTIFACTS_DIR / 'train_outputs'

for dir_path in (DRIVE_OUTPUT_DIR, ARTIFACTS_DIR, TRAIN_OUTPUT_DIR):
    dir_path.mkdir(parents=True, exist_ok=True)

print('Drive output root:', DRIVE_OUTPUT_DIR)
print('Baseline artifacts folder:', ARTIFACTS_DIR)
print('Repo:', REPO_DIR)

In [ ]:
# Install runtime dependencies for baseline-style evaluation in Colab via run()
run('python -m pip install -q --upgrade pip', cwd=REPO_DIR, stream=True)
run(
    'python -m pip install -q transformers accelerate sentencepiece huggingface_hub tqdm pandas scikit-learn packaging ninja psutil',
    cwd=REPO_DIR,
    stream=True,
)
run('python -m pip install --upgrade transformers', cwd=REPO_DIR, stream=True)

# FlashAttention docs recommend checking ninja works (`ninja --version` exits 0).
run('python -m pip uninstall -y ninja', cwd=REPO_DIR, stream=True, check=False)
run('python -m pip install -q ninja', cwd=REPO_DIR, stream=True)
ninja_check = run('ninja --version', cwd=REPO_DIR, stream=True, check=False)
if ninja_check.returncode != 0:
    raise RuntimeError(
        'ninja is installed but not functioning correctly. '
        'Please rerun: python -m pip uninstall -y ninja ; python -m pip install -q ninja'
    )
print('ninja check passed.')

# FlashAttention2 optional setup (recommended for speed/memory on A100).
# If installation fails, training will automatically fall back to SDPA in train.py.
'''
run('MAX_JOBS=4 python -m pip install -q flash-attn --no-build-isolation', cwd=REPO_DIR, stream=True, check=False)
run(
    "python -c \"import importlib.util; print('flash_attn installed' if importlib.util.find_spec('flash_attn') else 'flash_attn not available; SDPA fallback will be used')\"",
    cwd=REPO_DIR,
    stream=True,
    check=False,
)
'''

In [ ]:
# Optional: authenticate for gated models (e.g., Llama-2)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Step 1: prepare train/dev/test jsonl under benchmark/RAGTruth/baseline using run()
BASELINE_DIR = REPO_DIR / 'benchmark' / 'RAGTruth' / 'baseline'
print(f'Running data preparation from {BASELINE_DIR}...')
run('python prepare_dataset.py', cwd=BASELINE_DIR, stream=True)

# Verify data files were created
required_files = ['train.jsonl', 'dev.jsonl', 'test.jsonl']
for fname in required_files:
    fpath = BASELINE_DIR / fname
    if not fpath.exists():
        raise FileNotFoundError(
            f'Data preparation failed: {fname} not found in {BASELINE_DIR}. '
            'Check prepare_dataset.py output above for errors.'
        )
    size_mb = fpath.stat().st_size / (1024 * 1024)
    print(f'âœ?{fname}: {size_mb:.2f} MB')

print('\nData preparation complete. Files ready for training.')

In [ ]:
# Optional Step: train baseline model with tqdm progress bar via run()
from datetime import UTC, datetime
from huggingface_hub import hf_hub_download

BASELINE_DIR = REPO_DIR / 'benchmark' / 'RAGTruth' / 'baseline'
MODEL_NAME_OR_PATH = os.environ.get('RAGTRUTH_BASELINE_MODEL', 'Qwen/Qwen3-4B-Instruct-2507')

# Pre-check model access so failures are clear before long training startup.
try:
    hf_hub_download(repo_id=MODEL_NAME_OR_PATH, filename='config.json')
except Exception as exc:
    raise RuntimeError(
        'No access to selected HF model. Set a different public model in '
        'RAGTRUTH_BASELINE_MODEL and retry.'
    ) from exc

run_ts = datetime.now(UTC).strftime('%Y%m%d_%H%M%S')
train_output_dir = TRAIN_OUTPUT_DIR / f'baseline_{run_ts}'
train_output_dir.parent.mkdir(parents=True, exist_ok=True)

print('Starting baseline training. You should see tqdm progress from Hugging Face Trainer.')
print('Model:', MODEL_NAME_OR_PATH)
print('Training output dir:', train_output_dir)
print('FlashAttention2 is optional: train.py will fall back to SDPA if flash_attn is unavailable.')
print('Memory optimization: per_device_train_batch_size=1, gradient_accumulation_steps=16, model_max_length=1024')
run(
    "python train.py "
    f"--model_name_or_path {MODEL_NAME_OR_PATH} "
    f"--output_dir \"{train_output_dir}\" "
    "--do_train "
    "--num_train_epochs 1 "
    "--learning_rate 2e-05 "
    "--drop_neg_ratio -1 "
    "--train_file ./train.jsonl "
    "--eval_file ./dev.jsonl "
    "--bf16 True "
    "--tf32 True "
    "--use_flashatt_2 True "
    "--gradient_checkpointing True "
    "--per_device_train_batch_size 1 "
    "--per_device_eval_batch_size 1 "
    "--gradient_accumulation_steps 16 "
    "--model_max_length 1024 "
    "--logging_steps 1 "
    "--run_name baseline "
    "--lr_scheduler_type cosine "
    "--warmup_ratio 0.1 "
    "--save_steps 10000 "
    "--save_total_limit 2 "
    "--eval_strategy steps "
    "--eval_steps 80",
    cwd=BASELINE_DIR,
    stream=True,
 )

### Inference & Smoke Test (200 Samples)
By default, this notebook uses `MAX_SAMPLES = 200` for baseline inference. 

**For full results:**
Change `MAX_SAMPLES = 200` to `MAX_SAMPLES = None` in the execution cell below.

In [ ]:
import json
import re
import os
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM

BASELINE_DIR = REPO_DIR / 'benchmark' / 'RAGTruth' / 'baseline'
RAW_DATASET = BASELINE_DIR / 'test.jsonl'
OUTPUT_FILE = ARTIFACTS_DIR / 'prediction.colab.local.jsonl'

# TRAIN_OUTPUT_DIR is defined in the setup cell as ARTIFACTS_DIR / 'train_outputs'
MODEL_NAME_OR_PATH = os.environ.get('RAGTRUTH_BASELINE_MODEL', 'Qwen/Qwen3-4B-Instruct-2507')

if TRAIN_OUTPUT_DIR.exists():
    candidate_dirs = []
    # Scan for directories containing config.json or adapter_config.json
    for p in TRAIN_OUTPUT_DIR.glob("**/config.json"):
        candidate_dirs.append(p.parent)
    for p in TRAIN_OUTPUT_DIR.glob("**/adapter_config.json"):
        candidate_dirs.append(p.parent)
    
    candidate_dirs = list(set(candidate_dirs))
    if candidate_dirs:
        # Pick the most recently modified one
        newest_dir = max(candidate_dirs, key=lambda d: os.path.getmtime(d))
        MODEL_NAME_OR_PATH = str(newest_dir.absolute())
        print(f"Auto-detected locally trained model: {MODEL_NAME_OR_PATH}")
    else:
        print(f"No local models found in {TRAIN_OUTPUT_DIR}. Using default: {MODEL_NAME_OR_PATH}")
else:
    print(f"Training output directory not found. Using default: {MODEL_NAME_OR_PATH}")

MAX_SAMPLES = 200  # set None for full split
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.05
TOP_P = 0.95
TOP_K = 40

def export_metrics_and_snapshots(*, results: list[dict], output_file: Path, artifacts_dir: Path, model_name_or_path: str, max_samples: int | None) -> tuple[Path, Path, dict]:
    if not results:
        raise ValueError('No results available to export metrics. Run inference first.')

    df = pd.DataFrame.from_records(results)
    required_cols = {'labels', 'pred', 'task_type'}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns for metrics export: {missing}')

    df['is_halu'] = df['labels'].apply(lambda x: len(x) > 0)
    df['pred_halu'] = df['pred'].apply(lambda x: len(x.get('hallucination list', [])) > 0)

    overall_recall = recall_score(df['is_halu'], df['pred_halu'])
    overall_precision = precision_score(df['is_halu'], df['pred_halu'])
    overall_f1 = f1_score(df['is_halu'], df['pred_halu'])

    print(f'Overall Case recall/precision/f1: {overall_recall:.3f}, {overall_precision:.3f}, {overall_f1:.3f}')

    metrics_payload = {
        'overall': {
            'recall': float(overall_recall),
            'precision': float(overall_precision),
            'f1': float(overall_f1),
            'num_samples': int(len(df)),
        },
        'per_task': {},
        'model_name_or_path': model_name_or_path,
        'max_samples': max_samples,
        'prediction_file': str(output_file),
    }

    for task in ['QA', 'Summary', 'Data2txt']:
        temp = df[df['task_type'] == task]
        if len(temp) == 0:
            continue
        task_recall = recall_score(temp['is_halu'], temp['pred_halu'])
        task_precision = precision_score(temp['is_halu'], temp['pred_halu'])
        task_f1 = f1_score(temp['is_halu'], temp['pred_halu'])
        print(f'{task} Case recall/precision/f1: {task_recall:.3f}, {task_precision:.3f}, {task_f1:.3f}')
        metrics_payload['per_task'][task] = {
            'recall': float(task_recall),
            'precision': float(task_precision),
            'f1': float(task_f1),
            'num_samples': int(len(temp)),
        }

    run_ts = datetime.now(UTC).strftime('%Y%m%d_%H%M%S')
    metrics_file = artifacts_dir / f'ragtruth_baseline_metrics_{run_ts}.json'
    predictions_sync_file = artifacts_dir / f'ragtruth_baseline_predictions_{run_ts}.jsonl'

    with open(metrics_file, 'w', encoding='utf-8') as f:
        json.dump(metrics_payload, f, ensure_ascii=False, indent=2)

    import shutil
    shutil.copy2(output_file, predictions_sync_file)

    print('Saved metrics JSON to:', metrics_file)
    print('Synced predictions JSONL to:', predictions_sync_file)
    return metrics_file, predictions_sync_file, metrics_payload

In [ ]:
TEMPLATES = {
    'QA': (
        'Below is a question:\n'
        '{question}\n\n'
        'Below are related passages:\n'
        '{reference}\n\n'
        'Below is an answer:\n'
        '{response}\n\n'
        'Your task is to determine whether the summary contains either or both of the following two types of hallucinations:\n'
        '1. conflict: instances where the summary presents direct contraction or opposition to the original news;\n'
        '2. baseless info: instances where the generated summary includes information which is not substantiated by or inferred from the original news. \n'
        'Then, compile the labeled hallucinated spans into a JSON dict, with a key "hallucination list" and its value is a list of hallucinated spans. If there exist potential hallucinations, the output should be in the following JSON format: {{"hallucination list": [hallucination span1, hallucination span2, ...]}}. Otherwise, leave the value as a empty list as following: {{"hallucination list": []}}.\n'
        'Output only valid JSON with key "hallucination list" and no extra text:'
    ),
    'Summary': (
        'Below is the original news:\n'
        '{reference}\n\n'
        'Below is a summary of the news:\n'
        '{response}\n'
        'Your task is to determine whether the summary contains either or both of the following two types of hallucinations:\n'
        '1. conflict: instances where the summary presents direct contraction or opposition to the original news;\n'
        '2. baseless info: instances where the generated summary includes information which is not substantiated by or inferred from the original news. \n'
        'Then, compile the labeled hallucinated spans into a JSON dict, with a key "hallucination list" and its value is a list of hallucinated spans. If there exist potential hallucinations, the output should be in the following JSON format: {{"hallucination list": [hallucination span1, hallucination span2, ...]}}. Otherwise, leave the value as a empty list as following: {{"hallucination list": []}}.\n'
        'Output only valid JSON with key "hallucination list" and no extra text:'
    ),
    'Data2txt': (
        'Below is a structured data in the JSON format:\n'
        '{reference}\n\n'
        'Below is an overview article written in accordance with the structured data:\n'
        '{response}\n\n'
        'Your task is to determine whether the summary contains either or both of the following two types of hallucinations:\n'
        '1. conflict: instances where the summary presents direct contraction or opposition to the original news;\n'
        '2. baseless info: instances where the generated summary includes information which is not substantiated by or inferred from the original news. \n'
        'Then, compile the labeled hallucinated spans into a JSON dict, with a key "hallucination list" and its value is a list of hallucinated spans. If there exist potential hallucinations, the output should be in the following JSON format: {{"hallucination list": [hallucination span1, hallucination span2, ...]}}. Otherwise, leave the value as a empty list as following: {{"hallucination list": []}}.\n'
        'Output only valid JSON with key "hallucination list" and no extra text:'
    ),
}

def build_prompt(sample):
    task_type = sample.get('task_type')
    mapping = {
        'QA': 'QA',
        'Summary': 'Summary',
        'Data2txt': 'Data2txt'
    }
    task_key = mapping.get(task_type, task_type)

    if task_key not in TEMPLATES:
        task_key = 'Summary'

    if task_key == 'QA':
        return TEMPLATES['QA'].format(
            question=sample.get('question', ''),
            reference=sample.get('reference', ''),
            response=sample.get('response', ''),
        )
    return TEMPLATES[task_key].format(
        reference=sample.get('reference', ''),
        response=sample.get('response', ''),
    )

def parse_prediction(text):
    text = text.strip()

    try:
        pred = json.loads(text)
        if isinstance(pred, dict) and 'hallucination list' in pred:
            return pred
    except Exception:
        pass

    for match in re.finditer(r'\{[\s\S]*?\}', text):
        try:
            pred = json.loads(match.group(0))
            if isinstance(pred, dict) and 'hallucination list' in pred:
                return pred
        except Exception:
            continue

    list_match = re.search(r'hallucination list[^\[]*(\[[\s\S]*?\])', text, flags=re.I)
    if list_match:
        try:
            hall_list = json.loads(list_match.group(1))
            if isinstance(hall_list, list):
                return {'hallucination list': hall_list}
        except Exception:
            pass

    return {'hallucination list': []}

def build_generation_input(tokenizer, prompt):
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a strict JSON generator. Reply with only a JSON object with key \"hallucination list\". "
                    "No explanations, no markdown, no reasoning."
                ),
            },
            {"role": "user", "content": prompt}
        ]
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
    return prompt

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_OR_PATH,
    torch_dtype='auto',
    device_map='auto',
    trust_remote_code=True,
)
model.eval()

In [ ]:
rows = []
with open(RAW_DATASET, 'r', encoding='utf-8') as f:
    for line in f:
        rows.append(json.loads(line))

if MAX_SAMPLES is not None:
    rows = rows[:MAX_SAMPLES]

print(f'Total samples to evaluate: {len(rows)}')
results = []
existing_by_id = {}

if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            payload = line.strip()
            if not payload:
                continue
            row = json.loads(payload)
            sample_id = str(row.get('id', ''))
            if sample_id:
                existing_by_id[sample_id] = row

    if len(existing_by_id) > len(rows):
        raise ValueError(
            f'Resume mismatch: existing predictions ({len(existing_by_id)}) exceed current target samples ({len(rows)}). '
            'Either increase MAX_SAMPLES or move/delete the previous OUTPUT_FILE.'
        )

    print(f'Found existing predictions: {len(existing_by_id)}')

results = list(existing_by_id.values())
rows_to_run = [row for row in rows if str(row.get('id', '')) not in existing_by_id]
print(f'Resume progress: {len(existing_by_id)} complete, {len(rows_to_run)} remaining')

def build_generation_input(prompt_text, tokenizer):
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template is not None:
        messages = [
            {
                'role': 'system',
                'content': (
                    'You are a strict JSON generator. Reply with only a JSON object with key "hallucination list". '
                    'No explanations, no markdown, no reasoning.'
                ),
            },
            {'role': 'user', 'content': prompt_text.strip()},
        ]
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f'[INST] {prompt_text.strip()} [/INST]'

for item in tqdm(rows_to_run):
    prompt_text = build_prompt(item)
    gen_input_text = build_generation_input(prompt_text, tokenizer)

    inputs = tokenizer(gen_input_text, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    gen_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    pred = parse_prediction(gen_text)

    enriched = dict(item)
    enriched['pred'] = pred
    enriched['raw_pred_text'] = gen_text
    results.append(enriched)

    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

if rows_to_run:
    print('Saved predictions to:', OUTPUT_FILE)
else:
    print('No new rows to run; using existing predictions from:', OUTPUT_FILE)

metrics_file, predictions_sync_file, _ = export_metrics_and_snapshots(
    results=results,
    output_file=OUTPUT_FILE,
    artifacts_dir=ARTIFACTS_DIR,
    model_name_or_path=MODEL_NAME_OR_PATH,
    max_samples=MAX_SAMPLES,
 )
print('Auto-export complete after resume/fresh inference.')

In [ ]:
# Optional re-export cell (uses shared helper; safe to rerun)
metrics_file, predictions_sync_file, metrics_payload = export_metrics_and_snapshots(
    results=results,
    output_file=OUTPUT_FILE,
    artifacts_dir=ARTIFACTS_DIR,
    model_name_or_path=MODEL_NAME_OR_PATH,
    max_samples=MAX_SAMPLES,
 )
print('Re-exported metrics JSON to:', metrics_file)
print('Re-exported predictions snapshot to:', predictions_sync_file)